# 🔬 Deep Dive Analysis - Hidden Insights

Bu notebook standart EDA'nın ötesine geçip daha derin insightlar arıyor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
OUTPUT_PATH = Path("../output")
df = pd.read_csv(OUTPUT_PATH / 'youtube_trending_cleaned.csv')
df['publishedAt'] = pd.to_datetime(df['publishedAt'])
df['trending_date'] = pd.to_datetime(df['trending_date'])

print(f"✅ Loaded {len(df):,} rows")

---
## 1. 🚀 Viral Velocity - Hangi İçerikler Daha Hızlı Trend Oluyor?

In [ ]:
# Days to trend analysis
# Negatif değerler = hatalı veri veya yayın öncesi trend (premiere)
df_valid_days = df[(df['days_to_trend'] >= 0) & (df['days_to_trend'] <= 30)].copy()

viral_speed = df_valid_days.groupby('category_name')['days_to_trend'].agg(['mean', 'median']).round(2)
viral_speed = viral_speed.sort_values('median')

print("🚀 Viral Velocity by Category (days to trend):")
print("="*50)
for cat, row in viral_speed.iterrows():
    speed_emoji = "⚡" if row['median'] <= 1 else "🏃" if row['median'] <= 3 else "🐢"
    print(f"{speed_emoji} {cat:25} | Median: {row['median']:.1f} days | Mean: {row['mean']:.1f} days")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Median days to trend by category
viral_speed['median'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Median Days to Trend by Category', fontsize=12)
axes[0].set_xlabel('Days')
axes[0].axvline(x=viral_speed['median'].mean(), color='red', linestyle='--', label='Overall Avg')
axes[0].legend()

# Distribution of days to trend
axes[1].hist(df_valid_days['days_to_trend'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Days to Trend', fontsize=12)
axes[1].set_xlabel('Days after publish')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Key insight
fastest = viral_speed['median'].idxmin()
slowest = viral_speed['median'].idxmax()
print(f"\n💡 INSIGHT: {fastest} trends fastest (median {viral_speed.loc[fastest, 'median']:.1f} days)")
print(f"💡 INSIGHT: {slowest} takes longest (median {viral_speed.loc[slowest, 'median']:.1f} days)")

In [ ]:
# Viral speed by country
viral_by_country = df_valid_days.groupby('country_name')['days_to_trend'].median().sort_values()

plt.figure(figsize=(10, 5))
viral_by_country.plot(kind='barh', color='teal')
plt.title('Median Days to Trend by Country', fontsize=12)
plt.xlabel('Days')
plt.tight_layout()
plt.show()

print(f"\n💡 INSIGHT: {viral_by_country.idxmin()} has fastest trending cycle")
print(f"💡 INSIGHT: {viral_by_country.idxmax()} has slowest trending cycle")

---
## 2. 📅 Seasonal Patterns - Yılın Ritmi

In [ ]:
# Monthly patterns by category
df['trending_month_name'] = df['trending_date'].dt.month_name()
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

# Focus on key categories
key_cats = ['Music', 'Gaming', 'Sports', 'Entertainment', 'News & Politics']

monthly_cat = df[df['category_name'].isin(key_cats)].groupby(
    ['trending_month_name', 'category_name']
).size().unstack(fill_value=0)

# Normalize to show seasonal pattern (not just volume)
monthly_cat_pct = monthly_cat.div(monthly_cat.sum(axis=0), axis=1) * 100
monthly_cat_pct = monthly_cat_pct.reindex(month_order)

plt.figure(figsize=(14, 6))
for cat in key_cats:
    plt.plot(monthly_cat_pct.index, monthly_cat_pct[cat], marker='o', linewidth=2, label=cat)

plt.title('Seasonal Patterns by Category (% of yearly total)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('% of Category Total')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.axhline(y=100/12, color='gray', linestyle='--', alpha=0.5, label='Expected if uniform')
plt.tight_layout()
plt.show()

In [ ]:
# Find peak months for each category
print("📅 Peak Months by Category:")
print("="*50)
for cat in df['category_name'].unique():
    cat_monthly = df[df['category_name'] == cat]['trending_month_name'].value_counts()
    peak_month = cat_monthly.idxmax()
    peak_pct = cat_monthly.max() / cat_monthly.sum() * 100
    print(f"{cat:25} → Peak: {peak_month:10} ({peak_pct:.1f}%)")

---
## 3. 🎯 Sleeper Hits - Düşük View, Yüksek Engagement

In [ ]:
# Find videos with below-median views but top 25% engagement
median_views = df['view_count'].median()
top_engagement_threshold = df['engagement_rate'].quantile(0.75)

sleeper_hits = df[
    (df['view_count'] < median_views) & 
    (df['engagement_rate'] > top_engagement_threshold)
].copy()

print(f"🎯 Found {len(sleeper_hits):,} 'Sleeper Hit' entries")
print(f"   (Below median views but top 25% engagement)")

# What categories are sleeper hits?
sleeper_cats = sleeper_hits['category_name'].value_counts(normalize=True) * 100
overall_cats = df['category_name'].value_counts(normalize=True) * 100

comparison = pd.DataFrame({
    'Sleeper Hits %': sleeper_cats,
    'Overall %': overall_cats
}).fillna(0)
comparison['Difference'] = comparison['Sleeper Hits %'] - comparison['Overall %']
comparison = comparison.sort_values('Difference', ascending=False)

print("\n📊 Categories Over-represented in Sleeper Hits:")
print(comparison.round(1))

In [ ]:
# Visualize sleeper hit categories
plt.figure(figsize=(12, 6))
comparison['Difference'].sort_values().plot(kind='barh', 
    color=['green' if x > 0 else 'red' for x in comparison['Difference'].sort_values()])
plt.title('Categories Over/Under-represented in "Sleeper Hits"', fontsize=12)
plt.xlabel('Difference from Overall %')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\n💡 INSIGHT: These categories have loyal, engaged audiences despite lower views")

---
## 4. 📝 Title & Tag Strategy Analysis

In [ ]:
# Title length vs performance
df['title_length_bucket'] = pd.cut(df['title_length'], 
    bins=[0, 30, 50, 70, 100, 200], 
    labels=['Very Short (0-30)', 'Short (30-50)', 'Medium (50-70)', 'Long (70-100)', 'Very Long (100+)'])

title_perf = df.groupby('title_length_bucket').agg({
    'view_count': 'mean',
    'engagement_rate': 'mean',
    'video_id': 'count'
}).round(2)
title_perf.columns = ['avg_views', 'avg_engagement', 'count']

print("📝 Title Length vs Performance:")
print(title_perf)

In [ ]:
# Tag count vs performance
df['tag_bucket'] = pd.cut(df['tag_count'], 
    bins=[-1, 0, 5, 10, 20, 50, 100], 
    labels=['No tags', '1-5', '6-10', '11-20', '21-50', '50+'])

tag_perf = df.groupby('tag_bucket').agg({
    'view_count': 'mean',
    'engagement_rate': 'mean',
    'video_id': 'count'
}).round(2)
tag_perf.columns = ['avg_views', 'avg_engagement', 'count']

print("\n🏷️ Tag Count vs Performance:")
print(tag_perf)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Title length
title_perf['avg_views'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Average Views by Title Length')
axes[0].set_xlabel('Title Length')
axes[0].set_ylabel('Average Views')
axes[0].tick_params(axis='x', rotation=45)

# Tag count
tag_perf['avg_views'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Average Views by Tag Count')
axes[1].set_xlabel('Number of Tags')
axes[1].set_ylabel('Average Views')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 5. 🔒 Comments/Ratings Disabled Analysis

In [ ]:
# Who disables comments?
comments_disabled = df.groupby('category_name')['comments_disabled'].mean() * 100
ratings_disabled = df.groupby('category_name')['ratings_disabled'].mean() * 100

disabled_df = pd.DataFrame({
    'Comments Disabled %': comments_disabled,
    'Ratings Disabled %': ratings_disabled
}).sort_values('Comments Disabled %', ascending=False)

print("🔒 Disabled Features by Category:")
print(disabled_df.round(2))

In [ ]:
# Visualization
disabled_df.plot(kind='barh', figsize=(10, 8))
plt.title('% of Videos with Disabled Comments/Ratings by Category', fontsize=12)
plt.xlabel('Percentage')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Top categories for disabled
print(f"\n💡 INSIGHT: {comments_disabled.idxmax()} has most comments disabled ({comments_disabled.max():.1f}%)")

In [ ]:
# Which countries disable more?
country_disabled = df.groupby('country_name').agg({
    'comments_disabled': 'mean',
    'ratings_disabled': 'mean'
}) * 100

print("\n🌍 Disabled Features by Country:")
print(country_disabled.sort_values('comments_disabled', ascending=False).round(2))

---
## 6. 📈 K-POP Dominance Over Time

In [ ]:
# K-POP channels
kpop_channels = ['HYBE LABELS', 'BANGTANTV', 'JYP Entertainment', 'SMTOWN', 'BLACKPINK', 
                 'Big Hit Labels', 'Mnet K-POP', '1theK (원더케이)', 'Stone Music Entertainment']

df['is_kpop'] = df['channelTitle'].isin(kpop_channels)

# K-POP share over time
kpop_over_time = df.groupby('trending_year_month').agg({
    'is_kpop': 'mean'
}) * 100

kpop_over_time = kpop_over_time.reset_index()
kpop_over_time['date'] = pd.to_datetime(kpop_over_time['trending_year_month'])
kpop_over_time = kpop_over_time.sort_values('date')

plt.figure(figsize=(14, 5))
plt.plot(kpop_over_time['date'], kpop_over_time['is_kpop'], marker='o', markersize=3, linewidth=1, color='purple')
plt.fill_between(kpop_over_time['date'], kpop_over_time['is_kpop'], alpha=0.3, color='purple')
plt.title('K-POP Share of Trending Videos Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('% of Trending Videos')
plt.tight_layout()
plt.show()

print(f"\n📊 K-POP Statistics:")
print(f"   Overall share: {df['is_kpop'].mean()*100:.2f}%")
print(f"   Peak month: {kpop_over_time.loc[kpop_over_time['is_kpop'].idxmax(), 'trending_year_month']} ({kpop_over_time['is_kpop'].max():.1f}%)")

In [ ]:
# K-POP penetration by country
kpop_by_country = df.groupby('country_name')['is_kpop'].mean() * 100
kpop_by_country = kpop_by_country.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
kpop_by_country.plot(kind='barh', color='purple')
plt.title('K-POP Share of Trending by Country', fontsize=12)
plt.xlabel('% of Trending Videos')
plt.tight_layout()
plt.show()

print(f"\n💡 INSIGHT: {kpop_by_country.idxmax()} is most K-POP dominated ({kpop_by_country.max():.1f}%)")
print(f"💡 INSIGHT: {kpop_by_country.idxmin()} has least K-POP ({kpop_by_country.min():.1f}%)")

---
## 7. 📆 Weekday vs Weekend Patterns

In [ ]:
# Add weekend flag
df['is_weekend'] = df['trending_date'].dt.dayofweek >= 5

# Category distribution on weekdays vs weekends
weekday_cats = df[~df['is_weekend']]['category_name'].value_counts(normalize=True) * 100
weekend_cats = df[df['is_weekend']]['category_name'].value_counts(normalize=True) * 100

weekend_diff = weekend_cats - weekday_cats
weekend_diff = weekend_diff.sort_values()

plt.figure(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' for x in weekend_diff.values]
weekend_diff.plot(kind='barh', color=colors)
plt.title('Weekend vs Weekday Category Preferences', fontsize=12)
plt.xlabel('Difference (Weekend % - Weekday %)')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("💡 Green = More popular on weekends")
print("💡 Red = More popular on weekdays")

---
## 8. 🏆 Channel Concentration - Gini Analysis

In [ ]:
# How concentrated is trending? (Do few channels dominate?)
channel_counts = df['channelTitle'].value_counts()

# Top 1%, 5%, 10% of channels - what % of trending do they capture?
n_channels = len(channel_counts)
total_trending = channel_counts.sum()

top_1_pct = channel_counts.head(int(n_channels * 0.01)).sum() / total_trending * 100
top_5_pct = channel_counts.head(int(n_channels * 0.05)).sum() / total_trending * 100
top_10_pct = channel_counts.head(int(n_channels * 0.10)).sum() / total_trending * 100
top_20_pct = channel_counts.head(int(n_channels * 0.20)).sum() / total_trending * 100

print("🏆 Channel Concentration Analysis:")
print("="*50)
print(f"Total unique channels: {n_channels:,}")
print(f"\nTop 1% of channels ({int(n_channels*0.01):,}) capture {top_1_pct:.1f}% of trending")
print(f"Top 5% of channels ({int(n_channels*0.05):,}) capture {top_5_pct:.1f}% of trending")
print(f"Top 10% of channels ({int(n_channels*0.10):,}) capture {top_10_pct:.1f}% of trending")
print(f"Top 20% of channels ({int(n_channels*0.20):,}) capture {top_20_pct:.1f}% of trending")

print(f"\n💡 INSIGHT: Trending is {'highly concentrated' if top_10_pct > 50 else 'moderately distributed'}!")

In [ ]:
# Concentration by country
print("\n🌍 Channel Concentration by Country (Top 10% share):")
for country in df['country_name'].unique():
    country_channels = df[df['country_name'] == country]['channelTitle'].value_counts()
    n = len(country_channels)
    total = country_channels.sum()
    top_10 = country_channels.head(int(n * 0.10)).sum() / total * 100
    print(f"   {country:20}: Top 10% captures {top_10:.1f}%")

---
## 9. 🔥 Viral Outliers - Extreme Performers

In [ ]:
# Videos with extraordinary metrics
# Top 0.1% views
view_threshold = df['view_count'].quantile(0.999)
viral_outliers = df[df['view_count'] >= view_threshold].copy()

print(f"🔥 Viral Outliers (Top 0.1% by views): {len(viral_outliers):,} entries")
print(f"   View threshold: {view_threshold:,.0f}+ views")

# What characterizes viral outliers?
print("\n📊 Viral Outlier Characteristics:")
print(f"   Most common category: {viral_outliers['category_name'].mode()[0]}")
print(f"   Most common country: {viral_outliers['country_name'].mode()[0]}")
print(f"   Avg title length: {viral_outliers['title_length'].mean():.0f} chars")
print(f"   Avg tag count: {viral_outliers['tag_count'].mean():.0f} tags")
print(f"   Avg days to trend: {viral_outliers['days_to_trend'].median():.1f} days")

---
## 📝 Summary of Hidden Insights

In [ ]:
print("="*60)
print("📝 DEEP DIVE INSIGHTS SUMMARY")
print("="*60)
print("""
Document your key findings from this analysis:

1. VIRAL VELOCITY:
   - Fastest category: _______________
   - Slowest category: _______________

2. SEASONAL PATTERNS:
   - Key finding: _______________

3. SLEEPER HITS:
   - Categories with loyal audiences: _______________

4. TITLE/TAG STRATEGY:
   - Optimal title length: _______________
   - Optimal tag count: _______________

5. K-POP DOMINANCE:
   - Overall share: _______________
   - Most K-POP country: _______________

6. CHANNEL CONCENTRATION:
   - Top 10% capture: _______________% of trending

7. WEEKEND EFFECT:
   - Weekend favorites: _______________
   - Weekday favorites: _______________
""")